In [5]:
"""
Wikipedia Plot Scraper - Optimized & Production-Ready
Section 1.2 of the Master Engineering Specification

Features:
- Uses Wikipedia REST API via requests.Session
- Query format: "{movie_title} film {release_year}"
- Opensearch for exact page title resolution
- Fetches 'extract' from summary endpoint
- 0.3-second rate limiting (safe maximum)
- Smart skipping: movies with good TMDB overview (>=300 chars) are skipped
- Checkpoint saving every 100 movies
- Resume capability: never rescrape cached entries
- Year presence validation in plot text
- Detailed logging and failure categorization
"""

import time
import json
import requests
import pandas as pd
from pathlib import Path
from typing import Dict, Optional, List, Tuple

# ============================================================================
# CONFIGURATION - MUST BE CUSTOMIZED BEFORE FIRST RUN
# ============================================================================
USER_AGENT = "MovieRecommenderBot/1.0 (YOUR-REAL-EMAIL@example.com)"  # ← REPLACE WITH YOUR REAL EMAIL
SLEEP_TIME = 0.3              
CHECKPOINT_EVERY = 100         # Save progress every N movies
MIN_OVERVIEW_LEN = 300         # Skip scraping if TMDB overview already long enough
# YEAR_CHECK_CHARS = 400         # Check if year appears in first N chars of plot

OUTPUT_PATH = Path("../data/external/wikipedia_plots.json")
MOVIES_PATH = Path("../data/processed/movies_merged.csv")

# Wikipedia API endpoints
OPENSEARCH_URL = "https://en.wikipedia.org/w/api.php"
SUMMARY_URL = "https://en.wikipedia.org/api/rest_v1/page/summary/{title}"

# Ensure output directory exists
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": USER_AGENT})


# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================
def load_existing_cache() -> Dict[str, str]:
    """Load existing Wikipedia plots cache if present."""
    if OUTPUT_PATH.exists():
        try:
            with open(OUTPUT_PATH, 'r', encoding='utf-8') as f:
                cache = json.load(f)
            # Normalize keys and clean values
            return {str(k).strip(): (v.strip() if isinstance(v, str) else "") for k, v in cache.items()}
        except Exception as e:
            print(f"Error loading cache: {e}. Starting fresh.")
            return {}
    return {}


def save_checkpoint(cache: Dict[str, str]):
    """Save current cache state to disk."""
    with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
        json.dump(cache, f, ensure_ascii=False, indent=2)
    print(f"  [CHECKPOINT] Saved {len(cache)} entries to {OUTPUT_PATH}")


def resolve_page_title(movie_title: str, release_year: int) -> Optional[str]:
    """Resolve exact Wikipedia page title using opensearch (multiple attempts)."""
    # headers = {"User-Agent": USER_AGENT}
    query_attempts = [
        f"{movie_title} ({release_year} film)",
        f"{movie_title} film {release_year}",
        movie_title
    ]

    for attempt, query in enumerate(query_attempts, 1):
        try:
            response = session.get(
                OPENSEARCH_URL,
                params={
                    "action": "opensearch",
                    "search": query,
                    "limit": 1,
                    "namespace": 0,
                    "format": "json"
                },
                # headers=headers,
                timeout=10
            )
            response.raise_for_status()
            data = response.json()
            titles = data[1]
            if titles:
                print(f"  Resolved title (attempt {attempt}): {titles[0]}")
                return titles[0]
        except Exception as e:
            print(f"  Opensearch attempt {attempt} failed: {str(e)}")

    return None


def fetch_plot_summary(page_title: str) -> Optional[str]:
    """Fetch plot summary from Wikipedia REST summary endpoint."""
    url = SUMMARY_URL.format(title=page_title.replace(" ", "_"))
    # headers = {"User-Agent": USER_AGENT}

    try:
        response = session.get(url, timeout=10) #headers=headers,
        response.raise_for_status()
        data = response.json()
        extract = data.get("extract", "").strip()
        return extract if extract else None
    except Exception as e:
        time.sleep(1)
        print(f"  Summary fetch failed for '{page_title}': {str(e)}")
        return None


# ============================================================================
# MAIN SCRAPING LOGIC
# ============================================================================
def main():
    print("=" * 70)
    print("WIKIPEDIA PLOT SCRAPER - Optimized Version (Section 1.2 Compliant)")
    print("=" * 70)

    # Load movies
    print(f"\nLoading movies from: {MOVIES_PATH}")
    try:
        movies_df = pd.read_csv(MOVIES_PATH, low_memory=False)
        movies_df = movies_df[['id', 'title', 'release_year', 'overview', 'vote_count']].dropna(subset=['title', 'release_year'])
        movies_df['release_year'] = movies_df['release_year'].astype(int)
        movies_df['overview'] = movies_df['overview'].fillna("")
        print(f"Total movies loaded: {len(movies_df)}")
    except Exception as e:
        print(f"Error loading movies: {e}")
        return

        
    movies_df = movies_df[
        (movies_df['vote_count'] >= 50) 
    ]

    movies_df = movies_df.sort_values(
    by="vote_count",
    ascending=False
    )

    # Load existing cache
    wiki_plots = load_existing_cache()
    print(f"Already cached: {len(wiki_plots)} movies")

    # Filter movies needing scraping (exclude cached + those with good TMDB overview)
    movies_to_scrape = movies_df[
        (~movies_df['id'].astype(str).isin(wiki_plots.keys())) &
        (movies_df['overview'].str.len() < MIN_OVERVIEW_LEN)
    ]
    print(f"Effective movies to scrape after smart skip: {len(movies_to_scrape)}")

    if len(movies_to_scrape) == 0:
        print("\nAll movies are either cached or have sufficient TMDB overview. Done.")
        return

    successful = 0
    failed = 0
    failures_log: List[Tuple[str, str, int, str]] = []

    print("\n" + "=" * 70)
    print("STARTING SCRAPE")
    print("=" * 70)

    for i, row in enumerate(movies_to_scrape.itertuples(index=False), 1):
        tmdb_id = str(row.id)
        title = row.title.strip()
        year = int(row.release_year)
        # print(f"[{idx+1}/{len(movies_df)}] {title} ({year}) [TMDB: {tmdb_id}]")
        print(f"[{i}/{len(movies_to_scrape)}] {title} ({year}) [TMDB: {tmdb_id}]")

        # Resolve page title
        page_title = resolve_page_title(title, year)
        if not page_title:
            wiki_plots[tmdb_id] = ""
            failed += 1
            failures_log.append((tmdb_id, title, year, "No page title resolved"))
            # time.sleep(SLEEP_TIME)
            continue

        # Fetch plot
        plot_text = fetch_plot_summary(page_title)
        if not plot_text or len(plot_text) < 100:
            wiki_plots[tmdb_id] = ""
            failed += 1
            failures_log.append((tmdb_id, title, year, "Plot too short or empty"))
            # time.sleep(SLEEP_TIME)
            continue

        # # Year validation
        # if str(year) not in plot_text[:YEAR_CHECK_CHARS]:
        #     wiki_plots[tmdb_id] = ""
        #     failed += 1
        #     failures_log.append((tmdb_id, title, year, "Year not found in first 400 chars"))
        #     # time.sleep(SLEEP_TIME)
        #     continue

        # Success
        wiki_plots[tmdb_id] = plot_text
        successful += 1
        print(f"  ✓ Success ({len(plot_text)} chars)")

        time.sleep(SLEEP_TIME)

        # Checkpoint save
        if (successful + failed) % CHECKPOINT_EVERY == 0:
            save_checkpoint(wiki_plots)

    # Final save
    save_checkpoint(wiki_plots)

    # Summary
    print("\n" + "=" * 70)
    print("SCRAPING COMPLETE")
    print("=" * 70)
    print(f"Total cached entries: {len(wiki_plots)}")
    print(f"Successfully fetched this run: {successful}")
    print(f"Failed this run: {failed}")
    success_rate = (successful / len(movies_to_scrape)) * 100 if len(movies_to_scrape) > 0 else 0
    print(f"Success rate this run: {success_rate:.1f}%")
    print(f"(Spec expects ~60–70% overall success)")

    non_empty = sum(1 for v in wiki_plots.values() if v)
    print(f"\nTotal non-empty plots in cache: {non_empty}")
    print(f"Overall coverage: {(non_empty/len(wiki_plots))*100:.2f}%")

    if failures_log:
        print("\nSample failures (first 10):")
        for fail in failures_log[:10]:
            print(f"  - {fail[1]} ({fail[2]}): {fail[3]}")


if __name__ == "__main__":
    main()

WIKIPEDIA PLOT SCRAPER - Optimized Version (Section 1.2 Compliant)

Loading movies from: ..\data\processed\movies_merged.csv
Total movies loaded: 45338
Already cached: 0 movies
Effective movies to scrape after smart skip: 5031

STARTING SCRAPE
[1/5031] Inception (2010) [TMDB: 27205]
  Resolved title (attempt 1): Inception (2010 film)
  ✓ Success (580 chars)
[2/5031] Avatar (2009) [TMDB: 19995]
  Resolved title (attempt 1): Avatar (2009 film)
  ✓ Success (702 chars)
[3/5031] The Avengers (2012) [TMDB: 24428]
  Resolved title (attempt 1): The Avengers (2012 film)
  ✓ Success (750 chars)
[4/5031] Interstellar (2014) [TMDB: 157336]
  Resolved title (attempt 1): Interstellar (2014 film)
  ✓ Success (460 chars)
[5/5031] Django Unchained (2012) [TMDB: 68718]
  Resolved title (attempt 1): Django Unchained (2012 film)
  ✓ Success (703 chars)
[6/5031] Guardians of the Galaxy (2014) [TMDB: 118340]
  Resolved title (attempt 1): Guardians of the Galaxy (2014 film)
  ✓ Success (710 chars)
[7/5031] F